In [ ]:
from seedemu.layers import Base, Routing, Ebgp, Ibgp, Ospf, PeerRelationship
from seedemu.compiler import Docker, Platform
from seedemu.core import Emulator
from seedemu.utilities import Makers
import os
import sys
import networkx as nx
import re
import pickle
import time
from typing import List, Tuple, Dict
from seedemu.layers import Base, Ebgp, Routing, Ibgp, Ospf
from seedemu.layers.Ebgp import PeerRelationship
from seedemu.core import Binding, Filter, Emulator, Service, Router, AutonomousSystem
from typing import List, Tuple, Dict

def makeTransitAs(base: Base, asn: int, prefix: str, exchanges: List[int],
    intra_ix_links: List[Tuple[int, int]]) -> AutonomousSystem:
    """!
    @brief create a transit AS.

    @param base reference to the base layer.
    @param asn ASN of the newly created AS.
    @param exchanges list of IXP IDs to join.
    @param intra_ix_links list of tuple of IXP IDs, to create intra-IX links at.

    @returns transit AS object.
    """

    transit_as = base.createAutonomousSystem(asn)
    transit_as.setSubnets(prefix)
    routers: Dict[int, Router] = {}

    # Create a BGP router for each internet exchange (for peering purpose)
    for ix in exchanges:
        routers[ix] = transit_as.createRouter('r{}'.format(ix))
        routers[ix].joinNetwork('ix{}'.format(ix))

    # For each pair, create an internal network to connect the BGP routers
    # from two internet exchanges. There is no need to create a full-mesh
    # network among the BGP routers. As long as they can reach each other
    # over a single or multiple hops, it is OK.
    for (a, b) in intra_ix_links:
        name = 'net_{}_{}'.format(a, b)

        transit_as.createNetwork(name)
        routers[a].joinNetwork(name)
        routers[b].joinNetwork(name)

    return transit_as

def load_topology_data(filename: str) -> dict:
    """从文件加载拓扑数据（兼容 key: value 格式）"""
    with open(filename, 'r') as f:
        content = f.read().strip()
    
    # 处理格式：添加外层大括号，替换冒号为冒号+引号，处理列表
    # 1. 替换 key: 为 'key':
    content = re.sub(r'^(\w+):', r'"\1":', content, flags=re.MULTILINE)
    # 2. 每行末尾添加逗号（最后一行除外）
    lines = content.split('\n')
    lines = [line + ',' for line in lines[:-1]] + [lines[-1]] if lines else []
    content = '\n'.join(lines)
    # 3. 包裹成字典
    content = '{' + content + '}'
    
    # 安全解析为字典
    try:
        return eval(content)  # 此处使用eval是因为处理后的格式已符合Python字典规范
    except Exception as e:
        raise ValueError(f"解析拓扑数据失败: {e}")

dumpfile=None
hosts_per_as=2
###############################################################################
# 设置平台信息
platform = Platform.ARM64

# 加载拓扑数据
try:
    TOPOLOGY_DATA = load_topology_data('real_topology.txt')
except FileNotFoundError:
    print("错误: 未找到real_topology.txt文件")
    sys.exit(1)
except Exception as e:
    print(f"解析拓扑数据出错: {e}")
    sys.exit(1)

with open("assignment.pkl", "rb") as f:
    assignment = pickle.load(f)
emu   = Emulator()
ebgp  = Ebgp()
base  = Base()

###############################################################################
# 收集Transit AS的连接信息
# transit_info = {}
# for asn in TOPOLOGY_DATA["transit_asns"]:
#     asn = assignment[asn]['asn']
#     connected_ixs = set()
#     intra_links = []
#     for (ix_a, ix_b, t_asn, _) in TOPOLOGY_DATA["ix_ix_transit_edges"]:
#         ix_a = assignment[ix_a]['asn']
#         ix_b = assignment[ix_b]['asn']
#         t_asn = assignment[t_asn]['asn']
#         if t_asn == asn:
#             connected_ixs.add(ix_a)
#             connected_ixs.add(ix_b)
#             intra_links.append((ix_a, ix_b))
#     transit_info[asn] = (sorted(connected_ixs), intra_links)
# with open("transit_info.pkl", "wb") as f:
#     pickle.dump(transit_info, f)
with open("transit_info.pkl", "rb") as f:
    transit_info = pickle.load(f)

NameError: name '__file__' is not defined

In [ ]:
# 创建Transit Autonomous Systems
for asnuber in TOPOLOGY_DATA["transit_asns"]:
    asn = assignment[asnuber]['asn']
    prefix = assignment[asnuber]['ipv4']
    exchanges, intra_links = transit_info[asn]
    # temp=find_maximal_cliques(intra_links)
    # links=[]
    # for clique in temp:
    #     links.extend([(clique[i], clique[(i + 1) % len(clique)]) for i in range(len(clique))])
    links=[]
    clique=sorted(exchanges)
    links.extend([(clique[i], clique[(i + 1)]) for i in range(len(clique)-1)])
    makeTransitAs(base, asn, prefix, exchanges, list(set(links)))
    print(f"创建Transit AS{asn}: 连接IXP{exchanges}, 内部链路{links}")

###############################################################################
# 创建Stub AS并添加主机
stub_ix_map = {}
for (provider, customer, ix, rel) in TOPOLOGY_DATA["as_as_ix_edges"]:
    if rel == -1 and customer in TOPOLOGY_DATA["stub_asns"]:
        stub_ix_map[customer] = ix

for stub_asn in TOPOLOGY_DATA["stub_asns"]:
    ix = stub_ix_map[stub_asn]
    Makers.makeStubAsWithHosts(emu, base, stub_asn, ix, hosts_per_as)
    print(f"创建Stub AS{stub_asn}: 连接IXP{ix}, 主机数量{hosts_per_as}")

###############################################################################
# 配置RS多边对等关系
# ix_as_map = {assignment[ix]['asn']: [] for ix in TOPOLOGY_DATA["ixps"]}
# # 添加Transit AS
# for asn in TOPOLOGY_DATA["transit_asns"]:
#     asn = assignment[asn]['asn']
#     for ix in transit_info[asn][0]:
#         ix_as_map[ix].append(asn)
# # 添加Stub AS
# for stub_asn, ix in stub_ix_map.items():
#     ix_as_map[ix].append(stub_asn)

# # 应用RS对等配置
# for ix, asns in ix_as_map.items():
#     if asns:
#         ebgp.addRsPeers(ix, asns)
#         print(f"IXP{ix}配置RS多边对等: 参与AS{asns}")

###############################################################################
# 配置私有对等关系
for (a, b, ix, rel) in TOPOLOGY_DATA["as_as_ix_edges"]:
    a = assignment[a]['asn']
    b = assignment[b]['asn']
    ix = assignment[ix]['asn']
    rel = int(rel)
    # 转换关系: -1→Provider, 0→Peer
    if rel == -1:
        relationship = PeerRelationship.Provider
    elif rel == 0:
        relationship = PeerRelationship.Peer
    else:
        raise ValueError(f"无效关系值: {rel} (仅支持-1和0)")
    
    ebgp.addPrivatePeerings(ix, [a], [b], relationship)
    print(f"IXP{ix}配置私有对等: AS{a}与AS{b} (关系: {rel})")
t=time.time()
print("开始编译仿真网络...")
######################################################
# #########################
# 添加所有层到仿真器
emu.addLayer(base)
emu.addLayer(Routing())
emu.addLayer(ebgp)
emu.addLayer(Ibgp())
emu.addLayer(Ospf())

###############################################################################
# 输出结果
if dumpfile is not None:
    # 保存到文件供其他仿真器使用
    emu.dump(dumpfile)
else:
    emu.render()
    
    # 附加Internet Map容器并编译
    docker = Docker(platform=platform)
    emu.compile(docker, './output', override=True)
    print("仿真网络编译完成，输出目录: ./output")
print(f"编译时间: {time.time()-t} 秒")


In [ ]:
###############################################################################
# 创建Transit Autonomous Systems
for asn in TOPOLOGY_DATA["transit_asns"]:
    asn = assignment[asn]['asn']
    prefix = assignment[asn]['ipv4']
    exchanges, intra_links = transit_info[asn]
    # temp=find_maximal_cliques(intra_links)
    # links=[]
    # for clique in temp:
    #     links.extend([(clique[i], clique[(i + 1) % len(clique)]) for i in range(len(clique))])
    links=[]
    clique=sorted(exchanges)
    links.extend([(clique[i], clique[(i + 1)]) for i in range(len(clique)-1)])
    makeTransitAs(base, asn, prefix, exchanges, list(set(links)))
    print(f"创建Transit AS{asn}: 连接IXP{exchanges}, 内部链路{links}")

###############################################################################
# 创建Stub AS并添加主机
stub_ix_map = {}
for (provider, customer, ix, rel) in TOPOLOGY_DATA["as_as_ix_edges"]:
    if rel == -1 and customer in TOPOLOGY_DATA["stub_asns"]:
        stub_ix_map[customer] = ix

for stub_asn in TOPOLOGY_DATA["stub_asns"]:
    ix = stub_ix_map[stub_asn]
    Makers.makeStubAsWithHosts(emu, base, stub_asn, ix, hosts_per_as)
    print(f"创建Stub AS{stub_asn}: 连接IXP{ix}, 主机数量{hosts_per_as}")

###############################################################################
# 配置RS多边对等关系
ix_as_map = {ix: [] for ix in TOPOLOGY_DATA["ixps"]}
# 添加Transit AS
for asn in TOPOLOGY_DATA["transit_asns"]:
    asn = assignment[asn]['asn']
    for ix in transit_info[asn][0]:
        ix_as_map[ix].append(asn)
# 添加Stub AS
for stub_asn, ix in stub_ix_map.items():
    ix_as_map[ix].append(stub_asn)

# 应用RS对等配置
for ix, asns in ix_as_map.items():
    if asns:
        ebgp.addRsPeers(ix, asns)
        print(f"IXP{ix}配置RS多边对等: 参与AS{asns}")

###############################################################################
# 配置私有对等关系
for (a, b, ix, rel) in TOPOLOGY_DATA["as_as_ix_edges"]:
    # 转换关系: -1→Provider, 0→Peer
    if rel == -1:
        relationship = PeerRelationship.Provider
    elif rel == 0:
        relationship = PeerRelationship.Peer
    else:
        raise ValueError(f"无效关系值: {rel} (仅支持-1和0)")
    
    ebgp.addPrivatePeerings(ix, [a], [b], relationship)
    print(f"IXP{ix}配置私有对等: AS{a}与AS{b} (关系: {rel})")

######################################################
# #########################
# 添加所有层到仿真器
emu.addLayer(base)
emu.addLayer(Routing())
emu.addLayer(ebgp)
emu.addLayer(Ibgp())
emu.addLayer(Ospf())

###############################################################################
# 输出结果
if dumpfile is not None:
    # 保存到文件供其他仿真器使用
    emu.dump(dumpfile)
else:
    emu.render()
    
    # 附加Internet Map容器并编译
    docker = Docker(platform=platform)
    emu.compile(docker, './output', override=True)
    print("仿真网络编译完成，输出目录: ./output")


In [ ]:
    # 配置RS多边对等关系
    ix_as_map = {assignment[ix]['asn']: [] for ix in TOPOLOGY_DATA["ixps"]}
    # 添加Transit AS
    for asn in TOPOLOGY_DATA["transit_asns"]:
        asn = assignment[asn]['asn']
        for ix in transit_info[asn][0]:
            ix_as_map[ix].append(asn)
    # 添加Stub AS
    for stub_asn, ix in stub_ix_map.items():
        ix_as_map[ix].append(stub_asn)
    
    # 应用RS对等配置
    for ix, asns in ix_as_map.items():
        if asns:
            ebgp.addRsPeers(ix, asns)
            print(f"IXP{ix}配置RS多边对等: 参与AS{asns}")